In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
import pandas as pd

In [ ]:
df=pd.read_csv('C:/Users/Lenovo/Downloads/data.csv')

In [ ]:
df['date'] = pd.to_datetime(df['date'])
print(df['date'].dt.year.value_counts())
df=df.drop(['street','date'], axis =1)

date
2014    4600
Name: count, dtype: int64


In [ ]:
X=df.drop('price', axis=1)
y=df['price']

In [ ]:
from sklearn.model_selection import train_test_split
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=5
)
 

In [ ]:
numeric_cols = ['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors',
                'waterfront', 'view', 'condition', 'sqft_above', 'sqft_basement',
                'yr_built', 'yr_renovated']
categorical_cols = ['city', 'statezip', 'country']

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
])
pipe = Pipeline([
    ('pre', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42)),
])

In [ ]:
baseline_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', DecisionTreeRegressor(max_depth=5, random_state=42)),
])
baseline_pipe.fit(X_train_raw, y_train)
baseline_test_r2 = baseline_pipe.score(X_test_raw, y_test)
 

In [ ]:
from sklearn.model_selection import GridSearchCV
param_grid = {
    'model__max_depth': [3, 5, 7, 10],
    'model__min_samples_split': [20, 40, 60, 100],
    'model__min_samples_leaf': [10, 20, 30, 50],
    'model__max_features': [None, 'sqrt', 'log2'],
}
 
grid = GridSearchCV(
    pipe, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid.fit(X_train_raw, y_train)
 
print("Best Parameters:", grid.best_params_)
print("Best CV R2:", grid.best_score_)
 
best_pipe = grid.best_estimator_
 

Fitting 5 folds for each of 192 candidates, totalling 960 fits
Best Parameters: {'model__max_depth': 10, 'model__max_features': None, 'model__min_samples_leaf': 10, 'model__min_samples_split': 60}
Best CV R2: 0.3113815741215264


In [ ]:
train_r2 = best_pipe.score(X_train_raw, y_train)
test_r2 = best_pipe.score(X_test_raw, y_test)
 
print("\n--- Final results ---")
print("Baseline (untuned, max_depth=5) Test R2:", baseline_test_r2)
print("Tuned pipeline CV R2 (best_score_):", grid.best_score_)
print("Tuned pipeline Train R2:" , train_r2)
print("Tuned pipeline Test R2: ", test_r2) 


--- Final results ---
Baseline (untuned, max_depth=5) Test R2: 0.48729592186649295
Tuned pipeline CV R2 (best_score_): 0.3113815741215264
Tuned pipeline Train R2: 0.25508071209232297
Tuned pipeline Test R2:  0.45455962500585834
